In [ ]:
# Step 9: Self-Learning and Fallback Handling

## Objective
# Implement feedback collection, confidence-based fallback handling,
# and a mechanism for improving chatbot performance using user feedback.

# ## Key Tasks
# 1. Record user feedback.
# 2. Identify low-confidence intent predictions.
# 3. Generate fallback responses.
# 4. Store interaction history.
# 5. Prepare feedback data for future model improvement.

# ## Tools
# - Python
# - Pandas
# - Scikit-learn
# - Joblib
# - Datetime

In [1]:
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime

print("Self-learning and fallback libraries imported successfully.")

Self-learning and fallback libraries imported successfully.


In [2]:
# Set project paths and load the intent model

PROJECT_ROOT = Path(r"D:\Dynamic_AI_Chatbot_Project")

MODEL_DIR = PROJECT_ROOT / "models" / "nlp"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "self_learning"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "intent_logistic_regression.pkl"

if not model_path.exists():
    raise FileNotFoundError(
        f"Intent model not found: {model_path}"
    )

intent_model = joblib.load(model_path)

print("Intent recognition model loaded successfully.")
print("Model location:", model_path)

Intent recognition model loaded successfully.
Model location: D:\Dynamic_AI_Chatbot_Project\models\nlp\intent_logistic_regression.pkl


In [3]:
# Build confidence-based intent prediction

def predict_intent_with_fallback(message, threshold=0.45):

    probabilities = intent_model.predict_proba([message])[0]

    predicted_index = probabilities.argmax()
    confidence = probabilities[predicted_index]

    predicted_intent = intent_model.classes_[predicted_index]

    if confidence < threshold:
        return {
            "intent": "fallback",
            "confidence": float(confidence),
            "message": (
                "I'm not completely sure I understood your request. "
                "Could you please explain it in a different way?"
            )
        }

    return {
        "intent": predicted_intent,
        "confidence": float(confidence),
        "message": "Intent identified successfully."
    }


print("Confidence-based fallback function created successfully.")

Confidence-based fallback function created successfully.


In [5]:
# Test the fallback mechanism

test_messages = [
    "I want to book a flight to Paris.",
    "Can you tell me the weather in New York?",
    "Blah blah blah random text.",
    "I need help with my account.",
    "What is the meaning of life?"
]

for message in test_messages:

    result = predict_intent_with_fallback(message)

    print("USER:", message)
    print("INTENT:", result["intent"])
    print("CONFIDENCE:", round(result["confidence"], 3))
    print("SYSTEM:", result["message"])
    print("-" * 60)

test_messages = [
    "Hello, how are you?",
    "I want to speak with a human representative.",
    "The weather on Mars is strange and confusing."
]

for message in test_messages:

    result = predict_intent_with_fallback(message)

    print("USER:", message)
    print("INTENT:", result["intent"])
    print("CONFIDENCE:", round(result["confidence"], 3))
    print("SYSTEM:", result["message"])
    print("-" * 60)

USER: I want to book a flight to Paris.
INTENT: fallback
CONFIDENCE: 0.242
SYSTEM: I'm not completely sure I understood your request. Could you please explain it in a different way?
------------------------------------------------------------
USER: Can you tell me the weather in New York?
INTENT: fallback
CONFIDENCE: 0.17
SYSTEM: I'm not completely sure I understood your request. Could you please explain it in a different way?
------------------------------------------------------------
USER: Blah blah blah random text.
INTENT: fallback
CONFIDENCE: 0.093
SYSTEM: I'm not completely sure I understood your request. Could you please explain it in a different way?
------------------------------------------------------------
USER: I need help with my account.
INTENT: fallback
CONFIDENCE: 0.148
SYSTEM: I'm not completely sure I understood your request. Could you please explain it in a different way?
------------------------------------------------------------
USER: What is the meaning of life

In [6]:
def collect_user_feedback(message, predicted_intent, confidence,
                          feedback, corrected_intent=None):

    feedback_record = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_message": message,
        "predicted_intent": predicted_intent,
        "confidence": confidence,
        "user_feedback": feedback,
        "corrected_intent": corrected_intent
    }

    return feedback_record


print("User feedback collection function created successfully.")

User feedback collection function created successfully.


In [7]:
sample_feedback = collect_user_feedback(
    message="I need help with my internet connection.",
    predicted_intent="request_support",
    confidence=0.82,
    feedback="Incorrect",
    corrected_intent="complaint"
)

print(sample_feedback)

{'timestamp': '2026-11-22 19:38:40', 'user_message': 'I need help with my internet connection.', 'predicted_intent': 'request_support', 'confidence': 0.82, 'user_feedback': 'Incorrect', 'corrected_intent': 'complaint'}


In [8]:
# Initialize the interaction history


interaction_history = []

def log_interaction(message, intent_result, chatbot_response,
                    user_feedback="Not provided"):

    interaction = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "user_message": message,
        "predicted_intent": intent_result["intent"],
        "confidence": intent_result["confidence"],
        "chatbot_response": chatbot_response,
        "user_feedback": user_feedback
    }

    interaction_history.append(interaction)

    return interaction


print("Interaction history system created successfully.")

Interaction history system created successfully.


In [9]:
# Record sample chatbot interactions

sample_messages = [
    "Hello, I need some help.",
    "I want to speak with a human representative.",
    "Thank you for your support."
]

for message in sample_messages:

    result = predict_intent_with_fallback(message)

    if result["intent"] == "fallback":
        response = result["message"]
    else:
        response = "I understood your request. How can I help you?"

    log_interaction(
        message=message,
        intent_result=result,
        chatbot_response=response
    )

print("Sample interactions recorded successfully.")

display(pd.DataFrame(interaction_history))

Sample interactions recorded successfully.


,timestamp,user_message,predicted_intent,confidence,chatbot_response,user_feedback
0,2026-11-22 19:40:11,"Hello, I need some help.",fallback,0.246408,I'm not completely sure I understood your requ...,Not provided
1,2026-11-22 19:40:11,I want to speak with a human representative.,fallback,0.290128,I'm not completely sure I understood your requ...,Not provided
2,2026-11-22 19:40:11,Thank you for your support.,fallback,0.230534,I'm not completely sure I understood your requ...,Not provided


In [12]:
feedback_df = pd.DataFrame([sample_feedback])

feedback_path = OUTPUT_DIR / "user_feedback.csv"

if feedback_path.exists():
    feedback_df.to_csv(
        feedback_path,
        mode="a",
        header=False,
        index=False
    )
else:
    feedback_df.to_csv(feedback_path, index=False)

print("Feedback saved:", feedback_path)

Feedback saved: D:\Dynamic_AI_Chatbot_Project\outputs\self_learning\user_feedback.csv


In [13]:
# Save interaction history

interaction_df = pd.DataFrame(interaction_history)

interaction_path = OUTPUT_DIR / "interaction_history.csv"

interaction_df.to_csv(interaction_path, index=False)

print("Interaction history saved:", interaction_path)

Interaction history saved: D:\Dynamic_AI_Chatbot_Project\outputs\self_learning\interaction_history.csv


In [14]:
# Check output files

print("SELF-LEARNING OUTPUT VERIFICATION")
print("-" * 50)

for file_path in [feedback_path, interaction_path]:

    print("File:", file_path.name)
    print("Exists:", file_path.exists())

    if file_path.exists():
        print("Size:", file_path.stat().st_size, "bytes")

    print("-" * 50)

SELF-LEARNING OUTPUT VERIFICATION
--------------------------------------------------
File: user_feedback.csv
Exists: True
Size: 186 bytes
--------------------------------------------------
File: interaction_history.csv
Exists: True
Size: 668 bytes
--------------------------------------------------
